# Train AI 3D Reconstruction tren Google Colab

Notebook nay dung de train `project/src/training/training_pipeline.py` tren GPU Colab.

Chuan bi truoc:
- Nen project thanh `AI_3D_Reconstruction_Systerm.zip` va upload vao `MyDrive`.
- Trong zip/rar can co Pix3D raw dataset. Chap nhan ca `project/data/raw/pix3d.json` hoac `project/data/raw/pix3d/pix3d.json`.
- Bat GPU: `Runtime > Change runtime type > Hardware accelerator > GPU`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

PROJECT_ZIP = Path('/content/drive/MyDrive/AI_3D_Reconstruction_Systerm.zip')
PROJECT_RAR = Path('/content/drive/MyDrive/AI_3D_Reconstruction_Systerm.rar')
PROJECT_ON_DRIVE = Path('/content/drive/MyDrive/AI_3D_Reconstruction_Systerm')
PROJECT_DIR = Path('/content/AI_3D_Reconstruction_Systerm')

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if PROJECT_ZIP.exists():
    with zipfile.ZipFile(PROJECT_ZIP, 'r') as archive:
        archive.extractall('/content')
elif PROJECT_RAR.exists():
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'unrar'], check=True)
    subprocess.run(['unrar', 'x', '-o+', str(PROJECT_RAR), '/content/'], check=True)
elif PROJECT_ON_DRIVE.exists():
    shutil.copytree(PROJECT_ON_DRIVE, PROJECT_DIR)
else:
    raise FileNotFoundError(
        'Khong thay project. Hay upload zip/rar vao /content/drive/MyDrive/ '
        'hoac dat folder project tai /content/drive/MyDrive/AI_3D_Reconstruction_Systerm.'
    )

training_scripts = list(Path('/content').rglob('project/src/training/training_pipeline.py'))
if training_scripts:
    PROJECT_DIR = training_scripts[0].parents[3]
elif not PROJECT_DIR.exists():
    candidates = [p for p in Path('/content').glob('*AI_3D_Reconstruction*') if p.is_dir()]
    if candidates:
        PROJECT_DIR = candidates[0]

os.chdir(PROJECT_DIR)
print('Project dir:', PROJECT_DIR)
print('Current dir:', Path.cwd())


In [ ]:
!pip -q install -r project/requirements.txt


In [ ]:
import torch

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Dang chay CPU. Vao Runtime > Change runtime type de bat GPU.')


In [ ]:
import json
from collections import Counter
from pathlib import Path

pix3d_json_matches = sorted(Path.cwd().rglob('pix3d.json'))
assert pix3d_json_matches, 'Khong tim thay pix3d.json sau khi giai nen project.'

raw_dir = pix3d_json_matches[0].parent
project_dir = (Path.cwd() / 'project').resolve()
RAW_DIR_ARG = raw_dir.resolve().relative_to(project_dir).as_posix()

print('Detected raw_dir:', raw_dir)
print('Use --raw-dir:', RAW_DIR_ARG)

required_paths = [
    raw_dir / 'pix3d.json',
    raw_dir / 'img',
    raw_dir / 'mask',
    raw_dir / 'model',
]

for path in required_paths:
    print(path, 'OK' if path.exists() else 'MISSING')

assert all(path.exists() for path in required_paths), 'Thieu Pix3D raw dataset hoac sai cau truc thu muc.'

with (raw_dir / 'pix3d.json').open('r', encoding='utf-8') as file:
    items = json.load(file)

category_counts = Counter(item.get('category') for item in items)
print('Total metadata items:', len(items))
print('Categories:')
for category, count in sorted(category_counts.items()):
    print(f'  {category}: {count}')


## Chay thu pipeline

Cell nay train rat nhe de kiem tra dataset, import, GPU va output. Nen chay truoc khi train that.


In [ ]:
import subprocess
import sys

smoke_cmd = [
    sys.executable,
    'project/src/training/training_pipeline.py',
    '--raw-dir', RAW_DIR_ARG,
    '--output-dir', 'results/colab_smoke',
    '--categories', 'chair',
    '--max-samples', '64',
    '--num-points', '512',
    '--batch-size', '4',
    '--epochs', '2',
]

print('Running:', ' '.join(smoke_cmd))
subprocess.run(smoke_cmd, check=True)


## Train that

Goi y tren Colab Free: bat dau voi `NUM_POINTS = 1024`, `BATCH_SIZE = 4`, `EPOCHS = 50`. Neu bi het VRAM, giam `BATCH_SIZE` xuong 2 hoac giam `NUM_POINTS` xuong 512.


In [ ]:
import subprocess
import sys

CATEGORIES = ['chair']
MAX_SAMPLES = 500
NUM_POINTS = 1024
BATCH_SIZE = 4
EPOCHS = 50
LR = 1e-4
OUTPUT_DIR = 'results/colab_train'

cmd = [
    sys.executable,
    'project/src/training/training_pipeline.py',
    '--raw-dir', RAW_DIR_ARG,
    '--output-dir', OUTPUT_DIR,
    '--categories', *CATEGORIES,
    '--max-samples', str(MAX_SAMPLES),
    '--num-points', str(NUM_POINTS),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--lr', str(LR),
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

run_name = datetime.now().strftime('run_%Y%m%d_%H%M%S')
source_dir = Path('project') / OUTPUT_DIR
drive_results_dir = Path('/content/drive/MyDrive/reconstruction_results') / run_name
drive_results_dir.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(source_dir, drive_results_dir, dirs_exist_ok=True)

print('Saved results to:', drive_results_dir)
print('Checkpoint:', drive_results_dir / 'checkpoint' / 'transformer_pointcloud_net.pt')
print('Metrics:', drive_results_dir / 'metrics' / 'training_metrics.csv')
